# 03 — Perform analysis

Runs the analysis scripts in the order in which their results appear in the
paper. Prerequisites: `01_fetch_statsbomb_events.ipynb` and
`02_build_analysis_frame.ipynb` have been executed, so that
`data/events.parquet`, `data/lineups.parquet`, `data/analysis_frame.csv`,
and `data/odds_european.csv` exist.

Each script is self-contained and writes its outputs to `data/` (CSV tables)
or `figures/` (figures). A full run of this notebook re-estimates
every model in the paper, including the bound re-estimations, and takes
several hours on a laptop.

## Table 1 — Sample construction (Section 3.2)

In [1]:
!python src/build_attrition_table.py

analysis sample: 43,527 rows | treated 2,650
                      criterion  player_matches  dropped
Players listed in match lineups           89376        0
              Started the match           53651    35725
              Outfield position           48775     4876
 On the pitch through minute 60           44320     4455
       No yellow card in [0,15)           43831      489
  At most one yellow in [15,45]           43831        0
 Age and betting odds available           43527      304

matches: 2,440 | treated in final sample: 2,650 (6.09%)
reconciles with build_dml.load(): True
wrote data/attrition_table.csv


## Table 3 — Descriptive statistics by treatment status (Section 5.1)

In [2]:
!python src/build_descriptives_table.py

analysis sample: 43,527 rows | treated 2,650
    panel                                                 var  mean_unbooked  mean_booked   diff     p
covariate Opponent-directed defensive actions, minutes [0,15)          3.393        3.829  0.436 0.000
covariate     Ball-directed defensive actions, minutes [0,15)          1.643        1.794  0.151 0.000
covariate                          All events, minutes [0,15)         30.071       29.601 -0.469 0.075
covariate                           Score margin at minute 15          0.008       -0.028 -0.037 0.001
covariate                           Pre-match win probability          0.386        0.342 -0.044 0.000
covariate                                           Home team          0.506        0.460 -0.047 0.000
covariate                                         Age (years)         27.034       27.101  0.067 0.403
  outcome                                        opp_directed          3.481        3.466 -0.015 0.774
  outcome                   

## Table 4 — Main DML estimates (Section 5.2)

In [3]:
!python src/build_dml.py

analysis sample: 43,527 rows | treated 2,650
W: 62 columns
propensity range: (0.0202, 0.2235)

=== ATE per outcome ===
          dv  control_mean     ate     se   p   rel
opp_directed         3.481 -0.2119 0.0491 0.0 -6.1%
pressures 2.873 -0.1421 0.0415 0.0006 -4.9%
tackles 0.347 -0.0012 0.0126 0.9262 -0.3%
fouls 0.262 -0.0686 0.0096 0.0 -26.2%
ball_directed 1.779 -0.011 0.0288 0.7033 -0.6%
ball_recoveries 0.84 0.0026 0.0194 0.894 +0.3%
clearances 0.397 -0.0156 0.014 0.265 -3.9%
blocks 0.335 -0.0117 0.0117 0.3177 -3.5%
interceptions 0.207 0.0132 0.0106 0.2121 +6.4%

wrote data/dml_results.csv


## Figure 3 — Effects across outcome windows (Section 5.2)

In [4]:
!python src/build_multiwindow.py

analysis sample: 43,527 rows | treated 2,650
[45-50] n=46,553 (frame 43,527 + extras 3,026 after age-drop) treated=2,890
window              dv     n  treated  control_mean     ate     se      p    rel
 45-50       pressures 46553     2890         1.159 -0.0868 0.0231 0.0002  -7.5%
 45-50         tackles 46553     2890         0.128  0.0055 0.0075 0.4592  +4.3%
 45-50           fouls 46553     2890         0.099 -0.0279 0.0056 0.0000 -28.1%
 45-50   ball_directed 46553     2890         0.688 -0.0025 0.0171 0.8834  -0.4%
 45-50 ball_recoveries 46553     2890         0.327  0.0004 0.0117 0.9739  +0.1%
 45-50      clearances 46553     2890         0.151  0.0063 0.0081 0.4393  +4.2%
 45-50          blocks 46553     2890         0.130 -0.0155 0.0069 0.0235 -12.0%
 45-50   interceptions 46553     2890         0.080  0.0084 0.0063 0.1837 +10.4%
[45-60] n=44,706 (frame 43,251 + extras 1,455 after age-drop) treated=2,697
window              dv     n  treated  control_mean     ate     se      p 

In [5]:
!python src/build_multiwindow_figure.py

wrote figures/fig_multiwindow.png


## In-text — Withdrawal and censoring shares (Section 5.2)

In [6]:
!python src/build_censoring_table.py

population: starters on pitch at end of H1 | treated 3,108 | control 49,194
HT withdrawal: treated 4.0% vs control 1.80% (raw) / 1.93% (cell-adjusted)

window treated control_adj   raw  diff_pp lee_trim
 45-50    4.6%        2.2%  2.0%      2.5     2.5%
 45-60   11.0%        5.8%  5.4%      5.2     5.6%
 45-70   20.1%       12.7% 12.1%      7.3     8.4%
 45-80   29.2%       20.8% 19.9%      8.3    10.5%

wrote data/censoring_by_window.csv


## Section 5.3 — Effect heterogeneity
Joint moderator model (Figure 4 inputs, Table 9, in-text game-state effects), the heterogeneity figure, and the CLAN response-quartile profiles (Table 5).

In [7]:
!python src/build_hte_joint.py

analysis sample: 43,527 rows | treated 2,650
treated by pos5: {'DefMid': 833, 'CentralDef': 618, 'WideDef': 546, 'Forward': 449, 'OffMid': 204}
[opp_directed] overall heterogeneity p=0.5701 | blocks: {'position': 0.2473, 'game_state': 0.2441, 'age': 0.823, 'odds': 0.774}
[pressures] overall heterogeneity p=0.5518 | blocks: {'position': 0.3629, 'game_state': 0.2229, 'age': 0.6344, 'odds': 0.8209}
[tackles] overall heterogeneity p=0.3165 | blocks: {'position': 0.1143, 'game_state': 0.8502, 'age': 0.5461, 'odds': 0.4023}
[fouls] overall heterogeneity p=0.0008 | blocks: {'position': 0.0029, 'game_state': 0.0017, 'age': 0.839, 'odds': 0.2743}

=== fouls: joint-model coefficients ===
   dv        term    coef     se      p
fouls       const -0.0659 0.0230 0.0041
fouls pos_WideDef  0.0248 0.0277 0.3706
fouls  pos_DefMid -0.0705 0.0242 0.0035
fouls  pos_OffMid  0.0029 0.0407 0.9426
fouls pos_Forward  0.0129 0.0292 0.6587
fouls gs_trailing  0.0762 0.0240 0.0015
fouls  gs_leading -0.0133 0.0233 

In [8]:
!python src/build_hte_figure.py

analysis sample: 43,527 rows | treated 2,650
wrote figures/fig_hte_joint.png


In [9]:
!python src/build_cate_profiles.py

analysis sample: 43,527 rows | treated 2,650
fitted CATEs (fouls): n=43,527 | mean -0.064 | range [-0.224, +0.051] | 5th-95th [-0.153, +0.023] | share<0 87.2% | control mean 0.262

quartile mean fitted CATE:  {1: -0.132, 2: -0.074, 3: -0.051, 4: 0.001}
                                                    var  most_responsive  least_responsive   p
                                   Position: CentralDef           0.1310            0.2540 0.0
                                      Position: WideDef           0.0080            0.3620 0.0
                                       Position: DefMid           0.7760            0.0000 0.0
                                       Position: OffMid           0.0440            0.1110 0.0
                                      Position: Forward           0.0420            0.2730 0.0
                                   Game state: trailing           0.0810            0.8870 0.0
                                      Game state: level           0.5020          

## Table 6 — Teammate spillover (Section 5.4)

In [10]:
!python src/build_spillover_effect.py

analysis sample: 43,527 rows | treated 2,650
unbooked sample: 40,877 | teammate-exposed 15,940 (39.0%)
          dv  control_mean     ate     se      p   rel
opp_directed         3.451 -0.0051 0.0303 0.8655 -0.1%
pressures 2.854 -0.0255 0.0272 0.3488 -0.9%
tackles 0.344 0.0048 0.006 0.4264 +1.4%
fouls 0.254 0.0113 0.005 0.0233 +4.5%
ball_directed 1.778 0.0046 0.0161 0.7733 +0.3%
ball_recoveries 0.831 0.0311 0.0097 0.0014 +3.7%
clearances 0.402 -0.0153 0.0081 0.0599 -3.8%
blocks 0.338 -0.009 0.0063 0.1524 -2.7%
interceptions 0.207 -0.0043 0.0051 0.4022 -2.1%

wrote data/spillover_effect.csv


## Sections 5.5.1–5.5.2 — Overlap and unconfoundedness
Overlap support, trimming ladder, and Cinelli–Hazlett robustness values.

In [11]:
!python src/build_robustness_numbers.py

analysis sample: 43,527 rows | treated 2,650
e(W) range: [0.0202, 0.2235] | medians booked 0.062 / unbooked 0.058 | 99th pct booked 0.123 / unbooked 0.113

full sample: ATE -0.2119 (se 0.0491, p 0.0000)
trim [0.001,0.999]: n=43,527 (0.0% removed) ATE -0.2119 (p 0.0000)
trim [0.01,0.99]: n=43,527 (0.0% removed) ATE -0.2119 (p 0.0000)
trim [0.02,0.98]: n=43,527 (0.0% removed) ATE -0.2119 (p 0.0000)
trim [0.05,0.95]: n=32,517 (25.3% removed) ATE -0.2077 (p 0.0002)

Cinelli-Hazlett: RV(estimate=0) 2.05% | RV(p=.05) 1.12%
benchmark (position): partial R2 outcome 2.81% | treatment 0.118%

wrote data/robustness_numbers.csv


## Table 7 — Spillover-free re-estimation (Section 5.5.3)

In [12]:
!python src/build_spillover_robustness.py

analysis sample: 43,527 rows | treated 2,650
sample: 27,587 (15,940 teammate-exposed controls dropped) | treated 2,650
                 dv  control_mean    ate     se      p   rel
post_n_opp_directed         3.451 -0.206 0.0525 0.0001 -6.0%
post_n_pressure 2.854 -0.145 0.0462 0.0017 -5.1%
post_n_tackle 0.344 0.0047 0.0125 0.7057 +1.4%
post_n_foul_committed 0.254 -0.0581 0.0094 0.0 -22.9%
post_n_ball_directed 1.778 -0.0052 0.0292 0.8576 -0.3%
post_n_ball_recovery 0.831 0.0111 0.0189 0.5585 +1.3%
post_n_clearance 0.402 -0.0193 0.0147 0.1871 -4.8%
post_n_block 0.338 -0.0187 0.0117 0.1108 -5.5%
post_n_interception 0.207 0.017 0.0103 0.0996 +8.2%

wrote data/spillover_robustness.csv


## Section 5.5.4 and appendix — Selection bounds
Conditional Lee bounds for the primary window (in-text), all windows (appendix figure), binarized outcomes (Table 8, Figure 5), and Imbens–Manski intervals.

In [13]:
!python src/build_lee_conditional.py

analysis sample: 43,527 rows | treated 2,650
per-cell Lee trim p(c):
  Defender|0       6.5%   (frame n=7,426)
  Defender|1       3.8%   (frame n=5,467)
  Defender|2       6.7%   (frame n=5,930)
  Forward|0        4.2%   (frame n=3,429)
  Forward|1        6.3%   (frame n=4,232)
  Forward|2        2.0%   (frame n=2,659)
  Midfielder|0     8.0%   (frame n=3,654)
  Midfielder|1     4.5%   (frame n=4,810)
  Midfielder|2     6.4%   (frame n=5,920)

=== conditional Lee bounds ===
          dv  control_mean     ate   p  cond_lo  cond_hi cond_lo_rel cond_hi_rel
opp_directed         3.481 -0.2119 0.0  -0.3937   0.1796      -11.3%       +5.2%
pressures 2.873 -0.1421 0.0006 -0.2865 0.1763 -10.0% +6.1%
tackles 0.347 -0.0012 0.9262 -0.0225 0.1059 -6.5% +30.5%
fouls 0.262 -0.0686 0.0 -0.0856 0.0164 -32.7% +6.3%
ball_directed 1.779 -0.011 0.7033 -0.1163 0.2038 -6.5% +11.5%
ball_recoveries 0.84 0.0026 0.894 -0.0477 0.1458 -5.7% +17.4%
clearances 0.397 -0.0156 0.265 -0.0373 0.1048 -9.4% +26.4%
blocks 0

In [14]:
!python src/build_lee_windows.py

analysis sample: 43,527 rows | treated 2,650
[45-50] n=46,553 treated=2,890 | trims 0.0-4.8%
45-50 opp_directed 46553 2890 1.387 -0.1103 0.0278 0.0001 -8.0% -0.1486 0.0135 0.0278 0.0276 -10.7% +1.0%
45-50 pressures 46553 2890 1.159 -0.0868 0.0231 0.0002 -7.5% -0.1166 0.0094 0.023 0.0226 -10.1% +0.8%
45-50 tackles 46553 2890 0.128 0.0055 0.0075 0.4592 +4.3% 0.0024 0.0384 0.0075 0.0074 +1.9% +30.0%
45-50 fouls 46553 2890 0.099 -0.0279 0.0056 0.0 -28.1% -0.0308 0.0018 0.0056 0.0055 -31.1% +1.9%
45-50 ball_directed 46553 2890 0.688 -0.0025 0.0171 0.8834 -0.4% -0.0222 0.0697 0.0171 0.0169 -3.2% +10.1%
45-50 ball_recoveries 46553 2890 0.327 0.0004 0.0117 0.9739 +0.1% -0.0097 0.05 0.0117 0.0115 -3.0% +15.3%
45-50 clearances 46553 2890 0.151 0.0063 0.0081 0.4393 +4.2% -0.0 0.0474 0.0081 0.008 -0.0% +31.5%
45-50 blocks 46553 2890 0.13 -0.0155 0.0069 0.0235 -12.0% -0.0179 0.0184 0.0069 0.0067 -13.8% +14.2%
45-50 interceptions 46553 2890 0.08 0.0084 0.0063 0.1837 +10.4% 0.0052 0.0372 0.0063 0.006

In [15]:
!python src/build_lee_binary.py

analysis sample: 43,527 rows | treated 2,650
[45-50] n=46,553 treated=2,890 | trims 0.0-4.8%
45-50 any_opp_directed 0.6432 -1.89 0.9 0.0361 -2.9% -3.68 -0.99 0.9 0.9 True
45-50 any_ball_directed 0.4784 0.05 0.95 0.9579 +0.1% -1.37 1.2 0.96 0.96 False
45-50 any_pressure 0.6235 -1.88 0.91 0.0398 -3.0% -3.74 -0.97 0.91 0.91 True
45-50 any_tackle 0.1171 0.45 0.63 0.4748 +3.9% 0.06 2.79 0.63 0.63 True
45-50 any_foul 0.0943 -2.57 0.52 0.0 -27.3% -2.85 -0.08 0.52 0.51 True
45-50 any_recovery 0.2738 -0.03 0.88 0.9684 -0.1% -0.81 1.89 0.88 0.88 False
45-50 any_clearance 0.127 0.98 0.67 0.1423 +7.7% 0.52 3.42 0.67 0.67 True
45-50 any_block 0.1196 -1.48 0.61 0.0142 -12.4% -1.69 1.05 0.61 0.6 False
45-50 any_interception 0.0762 0.58 0.54 0.2789 +7.7% 0.23 3.09 0.54 0.54 True
[45-60] n=44,706 treated=2,697 | trims 2.0-8.0%
45-60 any_opp_directed 0.8954 0.54 0.55 0.3304 +0.6% -4.12 1.18 0.55 0.56 False
45-60 any_ball_directed 0.7939 0.79 0.77 0.3082 +1.0% -3.95 1.92 0.77 0.78 False
45-60 any_pressur

In [16]:
!python src/build_im_cis.py

=== count outcomes: bounds + IM 95% CI (relative %) ===
window              dv    rel lee_lo_rel lee_hi_rel  ci_lo_rel  ci_hi_rel
 45-50    opp_directed  -8.0%     -10.7%      +1.0%      -14.0        4.2
 45-50       pressures  -7.5%     -10.1%      +0.8%      -13.3        4.0
 45-50         tackles  +4.3%      +1.9%     +30.0%       -7.7       39.5
 45-50           fouls -28.1%     -31.1%      +1.9%      -40.4       10.9
 45-50   ball_directed  -0.4%      -3.2%     +10.1%       -7.3       14.2
 45-50 ball_recoveries  +0.1%      -3.0%     +15.3%       -8.8       21.1
 45-50      clearances  +4.2%      -0.0%     +31.5%       -8.8       40.1
 45-50          blocks -12.0%     -13.8%     +14.2%      -22.5       22.6
 45-50   interceptions +10.4%      +6.5%     +46.3%       -6.5       59.5
 45-60    opp_directed  -6.8%     -11.8%      +4.7%      -14.1        6.9
 45-60       pressures  -5.4%     -10.6%      +5.8%      -12.9        8.1
 45-60         tackles  -1.5%      -7.4%     +29.6%     

In [17]:
!python src/build_bounds_binary_figure.py

wrote figures/fig_bounds_binary.png


In [18]:
!python src/build_bounds_figure.py

wrote figures/fig_bounds.png


## Appendix — Timing of cards, defensive actions, and substitutions

In [19]:
!python src/build_desc_minutes.py

matches: 2,440
yellow cards: 10,349 (4.24/match) | H1 36.9% / H2 63.1% | median 57'
substitutions: 14,296 (5.86/match) | at HT 1,026 (7.2%) | after 60' 75.2% | median 71'

per position (yellows | subs):
            yellows  subs
Defender       4349  2223
Forward        1892  5543
Midfielder     4108  6530

wrote figures/fig_timing_cards.png, figures/fig_timing_subs.png


In [20]:
!python src/build_event_freq_figure.py

analysis sample: 43,527 rows | treated 2,650
wrote figures/fig_event_freq.png


## Appendix — Evaluation of the estimation procedure
Out-of-fold nuisance diagnostics and calibration, the overlap figure, and the learner/seed/balance insensitivity checks (Table 10).

In [21]:
!python src/build_nuisance_eval.py

analysis sample: 43,527 rows | treated 2,650
e(W): AUC 0.5625 | Brier 0.05702 vs base-rate 0.05718
m(W) opp_directed: OOF R2 0.1493
m(W) pressures: OOF R2 0.1697
m(W) tackles: OOF R2 0.0233
m(W) fouls: OOF R2 0.0108
m(W) ball_directed: OOF R2 0.0744
m(W) ball_recoveries: OOF R2 0.0298
m(W) clearances: OOF R2 0.1532
m(W) blocks: OOF R2 0.0085
m(W) interceptions: OOF R2 0.0250

calibration by decile:
     pred     obs     n
q                      
0  0.0372  0.0439  4353
1  0.0451  0.0462  4353
2  0.0498  0.0549  4352
3  0.0538  0.0487  4353
4  0.0570  0.0574  4353
5  0.0604  0.0653  4352
6  0.0643  0.0597  4353
7  0.0691  0.0685  4352
8  0.0764  0.0763  4353
9  0.0954  0.0880  4353

wrote data/nuisance_metrics.csv, figures/fig_calibration.png


In [22]:
!python src/build_robustness_figures.py

analysis sample: 43,527 rows | treated 2,650
wrote figures/fig_overlap.png


In [23]:
!python src/build_insensitivity.py

analysis sample: 43,527 rows | treated 2,650
       learner              dv     ate     se      p
HGB (baseline)    opp_directed -0.2236 0.0491 0.0000
HGB (baseline)       pressures -0.1542 0.0414 0.0002
HGB (baseline)         tackles -0.0010 0.0126 0.9339
HGB (baseline)           fouls -0.0695 0.0096 0.0000
HGB (baseline)   ball_directed -0.0077 0.0288 0.7904
HGB (baseline) ball_recoveries  0.0051 0.0193 0.7934
HGB (baseline)      clearances -0.0142 0.0141 0.3114
HGB (baseline)          blocks -0.0118 0.0117 0.3138
HGB (baseline)   interceptions  0.0135 0.0106 0.2031
     learner              dv     ate     se      p
HGB (deeper)    opp_directed -0.2188 0.0489 0.0000
HGB (deeper)       pressures -0.1468 0.0413 0.0004
HGB (deeper)         tackles -0.0012 0.0126 0.9213
HGB (deeper)           fouls -0.0694 0.0096 0.0000
HGB (deeper)   ball_directed -0.0056 0.0288 0.8460
HGB (deeper) ball_recoveries  0.0058 0.0194 0.7627
HGB (deeper)      clearances -0.0145 0.0141 0.3012
HGB (deeper)     